### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json
from IPython.display import display, update_display, Markdown
from scraper import fetch_website_contents, fetch_website_links



In [3]:
load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key found in .env file")
    raise ValueError("No API Key found in .env file")
elif not api_key.startswith('sk-proj'):
    print("Inavlid API Key found. Does not start with 'sk-proj'")
    raise ValueError("Invlaid API Key found, please check format")
else:
    print("API Key found and is valid")

openai = OpenAI(api_key=api_key)

model = 'gpt-4o-mini'




API Key found and is valid


In [ ]:
#some may be absolute or relative URLs
links = fetch_website_links("https://www.ibm.com/uk-en")
links

['https://www.ibm.com/products/offers-and-discounts?lnk=hpls1uk',
 'https://www.ibm.com/products?lnk=hpls2uk',
 'https://uk.newsroom.ibm.com/ibm-2026-x-force-threat-index?lnk=hprc1uk',
 'https://www.ibm.com/products/cognos-analytics#pricing',
 'https://ibm.briefingsource.com/register/U9Y8DFW2?lnk=hprc3uk',
 'https://www.ibm.com/products/webmethods-hybrid-integration?lnk=hprc4uk',
 'https://www.ibm.com/case-studies/scuderia-ferrari?lnk=hpcs1us',
 'https://www.ibm.com/case-studies/avid-solutions-international?lnk=hpcs2us',
 'https://www.ibm.com/case-studies/pfizer?lnk=hpcs3us',
 'https://www.ibm.com/software?lnk=hpfp1us',
 'https://www.ibm.com/solutions/ai-agents?lnk=hpfp2us',
 'https://www.ibm.com/solutions/data-and-ai?lnk=hpfp3us',
 'https://www.ibm.com/solutions/automation?lnk=hpfp4us',
 'https://www.ibm.com/solutions/hybrid-cloud?lnk=hpfp5us',
 'https://www.ibm.com/solutions/ai-models?lnk=hpfp6us',
 'https://www.ibm.com/solutions/analytics?lnk=hpfp7us',
 'https://www.ibm.com/solution

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [7]:
#one shot prompt
link_system_prompt = """ 
You are provided with a list of links on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, Career or products page.
You must respond in JSON Format as in the example below:
{
    "links": [
        {"type": "about page", "url": "https://company.com/about"},
        {"type": "products page", "url": "https://company.com/products"},
    ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f""" 
    Here is a list of links on the website {url} - 
    please decide which of these are relevant web links for a brochure about the company, respond 
    with the full https in JSON Format.
    Do not include terms of service, privact, or email links.

    links (some might be relative links):
    """

    links = fetch_website_links(url)
    user_prompt +="\n".join(links)
    return user_prompt

In [9]:
get_links_user_prompt("https://www.ibm.com/uk-en")

' \n    Here is a list of links on the websitehttps://www.ibm.com/uk-en - \n    please decide which of these are relevant web links for a brochure about the company, respond \n    with the full https in JSON Format.\n    Do not include terms of service, privact, or email links.\n\n    links (some might be relative links):\n    https://www.ibm.com/products/offers-and-discounts?lnk=hpls1uk\nhttps://www.ibm.com/products?lnk=hpls2uk\nhttps://uk.newsroom.ibm.com/ibm-2026-x-force-threat-index?lnk=hprc1uk\nhttps://www.ibm.com/products/cognos-analytics#pricing\nhttps://ibm.briefingsource.com/register/U9Y8DFW2?lnk=hprc3uk\nhttps://www.ibm.com/products/webmethods-hybrid-integration?lnk=hprc4uk\nhttps://www.ibm.com/case-studies/scuderia-ferrari?lnk=hpcs1us\nhttps://www.ibm.com/case-studies/avid-solutions-international?lnk=hpcs2us\nhttps://www.ibm.com/case-studies/pfizer?lnk=hpcs3us\nhttps://www.ibm.com/software?lnk=hpfp1us\nhttps://www.ibm.com/solutions/ai-agents?lnk=hpfp2us\nhttps://www.ibm.com/

In [12]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model = model,
        messages = [
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}

    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [13]:
select_relevant_links("https://www.ibm.com/uk-en")

{'links': [{'type': 'about page',
   'url': 'https://www.ibm.com/uk-en/about?lnk=hpii1uk'},
  {'type': 'careers page', 'url': 'https://www.ibm.com/careers?lnk=hpii1us'},
  {'type': 'products page', 'url': 'https://www.ibm.com/products?lnk=hpls2uk'},
  {'type': 'solutions page',
   'url': 'https://www.ibm.com/solutions/data-and-ai?lnk=hpfp3us'},
  {'type': 'case studies page',
   'url': 'https://www.ibm.com/case-studies/scuderia-ferrari?lnk=hpcs1us'},
  {'type': 'case studies page',
   'url': 'https://www.ibm.com/case-studies/avid-solutions-international?lnk=hpcs2us'},
  {'type': 'case studies page',
   'url': 'https://www.ibm.com/case-studies/pfizer?lnk=hpcs3us'}]}

## Make the Brochure!!

In [14]:
# combine functions into one

def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])

    return result

In [15]:
fetch_page_and_all_relevant_links("https://www.ibm.com/uk-en")

'## Landing Page:\n\nIBM\n\nSave 30% on select IBM products\nShop deals on smart solutions – with multiple payment options available\nExplore the offers\nView all products\nRecommended for you\nIBM 2026 X-Force Threat Index\nAI-Driven Attacks are Escalating as Basic Security Gaps Leave Enterprises Exposed\nIBM Cognos Analytics\nSave 30% on your first monthly or annual subscription\nEvent\nAccelerating Business Transformation\nwebMethods Hybrid Integration\nTransform AI pilots into production with enterprise integration – in one unified iPaaS\nSmarter business. Real impact.\nScuderia Ferrari HP\nThe Ferrari brand is fueled by a relentless drive to innovate. They’re constantly evolving, testing new ideas and striving for peak performance. It’s this shared value of continuous innovation that makes the collaboration between Ferrari and IBM so powerful. To make that vision a reality, the F1 team partnered with IBM Consulting® to completely reimagine their mobile app.\n2x\nincrease in daily 

In [16]:
brochure_system_prompt = """ 
You are the assitant that anlyses the contents of several web relevant webpages from a company website and
creates a short, professional brochure about the company for prospective customers, investors, and employees.
Respond in markdown without code blocks.
Include details of company culture, customers, and careers if you have the information
"""

In [20]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f""" 
    You are looking at a company called: {company_name}
    Here are the contents of its landing page and other relevant pages.
    Use this information to build a short, professional brochure of the company in markdown without code blocks \n\n
    """
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt += user_prompt[:5_000]#truncate if over 5000 characters

    return user_prompt

In [21]:
get_brochure_user_prompt("IBM UK", "https://www.ibm.com/uk-en")

" \n    You are looking at a company called: IBM UK\n    Here are the contents of its landing page and other relevant pages.\n    Use this information to build a short, professional brochure of the company in markdown without code blocks \n\n\n    ## Landing Page:\n\nIBM\n\nSave 30% on select IBM products\nShop deals on smart solutions – with multiple payment options available\nExplore the offers\nView all products\nRecommended for you\nIBM 2026 X-Force Threat Index\nAI-Driven Attacks are Escalating as Basic Security Gaps Leave Enterprises Exposed\nIBM Cognos Analytics\nSave 30% on your first monthly or annual subscription\nEvent\nAccelerating Business Transformation\nwebMethods Hybrid Integration\nTransform AI pilots into production with enterprise integration – in one unified iPaaS\nSmarter business. Real impact.\nScuderia Ferrari HP\nThe Ferrari brand is fueled by a relentless drive to innovate. They’re constantly evolving, testing new ideas and striving for peak performance. It’s t

In [25]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=model,
        messages = [
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ]
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [26]:
create_brochure("IBM UK", "https://www.ibm.com/uk-en")

# IBM UK Brochure

## Company Overview
IBM (International Business Machines Corporation) is a global leader in technology and consulting solutions, committed to driving innovation in business and society. With a rich history spanning over a century, IBM has been at the forefront of technological advancements, continuously redefining the boundaries of what is possible.

### Our Mission
"We don’t just make business work better. We make the world work better."  
— Arvind Krishna, Chairman and CEO

IBM provides a comprehensive environment of services and technologies, including Consulting, Software, Infrastructure, and Ecosystem partnerships. Our goal is to empower organizations to overcome their challenges through intelligent solutions.

## Company Culture
At IBM, our culture is built on the principles of respect for the individual, innovation, and a commitment to ethical practices. Our employees, known as IBMers, are our greatest asset, driving progress through creativity and collaboration. Our diverse workforce reflects our commitment to inclusion, fostering an environment where every voice matters and innovation thrives.

### Core Values
- **Respect** for each individual
- **Dedication** to customer success
- **Innovation** that continuously pushes boundaries
- **Integrity** in all our actions

## Our Customers
IBM serves a wide range of clients, including industry leaders like Scuderia Ferrari, Pfizer, and Avid Solutions. Through targeted and intelligent solutions, we have helped our clients achieve significant operational efficiencies and drive business transformation:

- **Scuderia Ferrari**: Partnered to enhance fan engagement through innovative digital solutions, resulting in a 2x increase in daily active users of their mobile app.
- **Pfizer**: Leveraged IBM Power solutions for mission-critical SAP applications, enabling a 93% reduction in database footprint.
- **Avid Solutions**: Utilized IBM watsonx Orchestrate to automate processes, achieving a 25% reduction in customer onboarding time.

## Products and Services
IBM offers a wide array of products tailored to meet modern business needs:

- **Consulting**: Delivering strategic insights for meaningful business impact.
- **Cloud Solutions**: Rapid access to shared processing resources for agile operations.
- **Cybersecurity**: Comprehensive security solutions to empower organizations in uncertain environments.
- **Analytics**: Facilitating data interpretation to derive valuable insights.

### Limited-Time Offers
Take advantage of our special offers and discounts on select IBM products, such as:
- Save 30% on your first subscription to IBM Cognos Analytics
- Flexible payment options available for various solutions

## Careers at IBM
Define your career with IBM, where we believe in empowering individuals to achieve their full potential. We offer exciting opportunities in various fields, including Consulting, Software Engineering, Data & Analytics, and Cybersecurity. 

### Why Join IBM?
- Be part of an innovative team that collaborates to solve the world’s most pressing challenges.
- Engage in career development through programs like IBM SkillsBuild, providing valuable training and opportunities for underrepresented communities in technology.

Explore roles that match your interests and skills at [IBM Careers](https://www.ibm.com/careers).

## Join Us in Transforming the Future
At IBM, we are committed to leveraging technology for the greater good. Join us as we drive meaningful change and positively impact communities worldwide through our IBM Sustainability Accelerator and various initiatives aimed at fostering enterprise social good. 

**For more information, visit our [website](https://www.ibm.com)**.

In [28]:
# stream your results

def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [29]:
stream_brochure("IBM UK", "https://www.ibm.com/uk-en")

# IBM UK: Innovating Today for a Smarter Tomorrow

---

## About IBM UK

IBM UK stands at the forefront of technological innovation, blending cutting-edge research with real-world applications to make businesses and society better and more secure. Under the leadership of Chairman and CEO Arvind Krishna, IBM brings together a comprehensive ecosystem of consulting, software, infrastructure, and partnerships to solve complex challenges for clients worldwide.

IBM UK focuses on empowering clients with hybrid cloud, artificial intelligence, cybersecurity, and data analytics solutions—driving transformation across industries with a commitment to open-source innovation and sustainability.

---

## Our Solutions & Products

- **Analytics**: Harness meaningful data patterns to inform smarter decisions and optimize business outcomes.
- **Business Operations**: Streamline processes for greater productivity and operational efficiency.
- **Cloud Computing**: Access scalable, rapid on-demand cloud services tailored for hybrid cloud environments.
- **Cybersecurity**: Protect your enterprise with AI-infused, zero-trust security systems guarding every access point.
- **IT Infrastructure**: Servers, storage, and mainframe solutions designed to modernize and secure your critical systems.
- **Automation & AI**: Transform operations with AI-powered automation to boost resilience, agility, and growth.

Special offers include up to 30% savings on select products and flexible financing options to accelerate your initiatives.

---

## Impactful Client Partnerships

- **Scuderia Ferrari HP**: Reimagined fan engagement through IBM Consulting, doubling daily active users and increasing fan interaction time by 35% via a new mobile app.
- **Avid Solutions**: Automated key workflows like onboarding and project management using IBM watsonx Orchestrate, reducing errors by 10% and onboarding time by 25%, enhancing employee and customer satisfaction.
- **Pfizer**: Enabled scalable SAP workloads and streamlined digital transformation with IBM Power solutions, achieving a 93% database reduction and significant cost avoidance, accelerating delivery of critical medicines.

---

## Company Culture & Social Impact

IBM UK fosters a culture of continuous innovation, collaboration, and learning—united by a mission to make the world work better. Employees are empowered to think big, solve complex problems, and leverage technology to create meaningful change.

- **IBM SkillsBuild**: A free education program delivering career opportunities and skills development to underrepresented communities in technology.
- **IBM Sustainability Accelerator**: Driving social good with technology and expertise to support vulnerable communities worldwide.
- Ethical commitment to security and privacy is maintained through the IBM Trust Center, ensuring client and user trust in every solution.

---

## Careers at IBM UK

At IBM UK, your skills can help shape a better future. Join a team where technology meets human potential across diverse roles including:

- Consulting & Business Transformation
- Software Engineering & AI Development
- Cloud & Security Solutions
- Data Analytics & Research
- Project & Product Management

You’ll work with the latest technologies like hybrid cloud, AI/watsonx™, Red Hat open-source platforms, and cutting-edge infrastructure while being supported in a culture focused on growth, inclusion, and impact.

Explore personalized job matches and join a network committed to innovation, customer success, and societal progress.

---

## Why Choose IBM UK?

- **Trusted Partner**: Delivering faster, meaningful results through deep industry expertise.
- **Open Innovation**: Leveraging open-source and hybrid cloud for flexible, future-proof solutions.
- **AI & Data Leadership**: Empowering smarter business decisions with integrated data and AI platforms.
- **Security & Compliance**: Leading enterprise-grade security infused with AI to safeguard your systems.
- **Sustainability Focus**: Driving positive global change by combining technology with social responsibility.

---

## Contact & Explore

Unlock smarter business transformation with IBM UK. Discover special offers, in-depth solutions, client success stories, and career opportunities at [IBM UK Website](https://www.ibm.com/uk).

---

**IBM UK** — Smarter business. Real impact. Together, we make the world work better.